**Historical feature-MLP tutorial.** Use [the project walkthrough](../project_walkthrough.ipynb) for the supported selected models and end-to-end workflow. This tutorial requires its original prepared local table or its explicitly artificial demo.


# PPG feature MLP ? issue #29

The production experiment uses NeuroKit-cleaned PPG features and the team?s PR #41 complete-beat median ABP labels. The MLP predicts SBP and DBP. Test waveforms remain untouched.

For the predefined full comparison and feature importance, run `python -m experiments.runners.full_experiment --data-dir data/derived/nn_fullrun/20261029_pr41` after preparation and artifact verification. The cells below explain the individual training steps and can run a separate fresh experiment.

In [ ]:
from pathlib import Path
import json
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models/feature_mlp.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this archived tutorial inside the repository checkout.")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display

from models.feature_mlp import FeatureMLP
from models.train_feature_mlp import (
    DEMO_LABEL_VERSION, TrainConfig, file_sha256, fit_preprocessing,
    load_bundle, make_demo_data, make_loader, plot_history, predict,
    prepare_rows, read_feature_table, regression_metrics, run_experiment, transform_features,
)

torch.set_num_threads(2)
torch.manual_seed(2026)
print(f"PyTorch {torch.__version__}; device: CPU")

## 1. Choose the input

The default paths use the locally prepared real dataset. Run cleaning and artifact verification first. Set `USE_DEMO = True` only for an artificial programming example. The explicit configuration fixes the input feature order and label version; this notebook does not choose labels or read raw/test waveforms.

In [ ]:
USE_DEMO = False
DATA_PATH = ROOT / "data/derived/nn_fullrun/20261029_pr41/model_table.csv"
FEATURE_CONFIG_PATH = ROOT / "data/derived/nn_fullrun/20261029_pr41/feature_config_six.json"

if USE_DEMO:
    table, feature_names = make_demo_data(seed=2026)
    dataset_version = "artificial_v1"
    label_version = DEMO_LABEL_VERSION
    print("ARTIFICIAL DATA ONLY ? not a blood-pressure performance result")
else:
    definition = json.loads(FEATURE_CONFIG_PATH.read_text(encoding="utf-8"))
    feature_names = definition["feature_names"]
    dataset_version = definition["dataset_version"]
    label_version = definition["label_version"]
    table = read_feature_table(DATA_PATH)

print(f"{len(table):,} rows; {len(feature_names)} selected PPG features")
display(table[["window_id", "split"] + feature_names[:3] + ["sbp", "dbp"]].head())

## 2. Keep the saved splits and IDs

`prepare_rows` checks IDs, recording overlap, the explicit feature list, the label version, and usability. It keeps only eligible train/validation rows and records exclusions. It does not make a new random split or process the reserved test values.

Inspect one row: its feature values and pressure answers must refer to the same window.

In [ ]:
splits, row_log = prepare_rows(table, feature_names, label_version)
train = splits["train"]
validation = splits["validation"]
display(row_log.groupby("split")["included"].agg(total="size", included="sum"))
print("Example ID:", train.iloc[0]["window_id"])
print("Example answers [SBP, DBP]:", train.iloc[0][["sbp", "dbp"]].to_numpy())

## 3. Calculate training statistics, then inspect a batch

Here **fit** means calculate and remember statistics. A missing feature can use its training median; scaling subtracts a feature's training mean and divides by its training standard deviation. The saved numbers are reused for validation and future inputs.

We also standardize the two pressure answers for optimization. Reported predictions are converted back to mmHg. This does not change the teammate's stored answers.

A batch is a small group of rows processed together. The default contains at most 256 windows.

In [ ]:
settings = TrainConfig()
preparation = fit_preprocessing(train, feature_names)
loader = make_loader(train, preparation, settings)
x_batch, y_batch = next(iter(loader))
print("Input batch:", tuple(x_batch.shape))
print("Answer batch:", tuple(y_batch.shape))
print("First standardized answers:", y_batch[0].tolist())
assert x_batch.shape[1] == len(feature_names)
assert y_batch.shape[1] == 2

## 4. Write the layers

`Linear(F, 32)` combines the feature values into 32 learned values. ReLU makes the mapping nonlinear. Dropout disables some hidden outputs during training. The final two outputs are standardized SBP/DBP; there is no softmax or sigmoid.

**Your practice:** in a new cell, write these layers yourself and check the output shape. The editable reference below is a practice model. The complete saved run later uses the shared `FeatureMLP` class in `models/feature_mlp.py`; copy your finished layers there so the notebook and command line use the same implementation.

In [ ]:
practice_model = nn.Sequential(
    nn.Linear(len(feature_names), 32),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 2),
)
print(practice_model)
print("Output shape:", tuple(practice_model(x_batch).shape))
assert practice_model(x_batch).shape == y_batch.shape

## 5. Write one training step

The loss is a number describing how far predictions are from the answers. Backpropagation calculates how each weight contributes to that loss. The optimizer adjusts the weights. Clearing gradients prevents accidentally adding a previous batch's gradients.

**Your practice:** rewrite this short cell in your own words/code. A loss is not a percentage or a mmHg score here because the answers are standardized.

In [ ]:
optimizer = torch.optim.AdamW(practice_model.parameters(), lr=0.001, weight_decay=0.0001)
loss_fn = nn.MSELoss()
practice_model.train()
predictions = practice_model(x_batch)
loss = loss_fn(predictions, y_batch)
optimizer.zero_grad()
loss.backward()
optimizer.step()
print(f"One-batch training MSE: {loss.item():.4f} (standardized targets)")

## 6. Write an epoch and validation

An epoch is one pass through the training examples. This practice cell performs three epochs. `eval()` disables dropout during validation; `no_grad()` avoids calculating gradients. Validation never calls `backward()` or `optimizer.step()`.

**Your practice:** write the loop before looking at the reference. The production training loop is `train_epoch` in `models/train_feature_mlp.py`. Copy your finished loop there when you are comfortable.

In [ ]:
x_validation = torch.from_numpy(transform_features(validation, preparation))
validation_answers = validation[["sbp", "dbp"]].to_numpy(dtype=float)

for epoch in range(1, 4):
    practice_model.train()
    total_loss = 0.0
    for inputs, answers in loader:
        predictions = practice_model(inputs)
        loss = loss_fn(predictions, answers)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(inputs)

    practice_model.eval()
    with torch.no_grad():
        standardized = practice_model(x_validation).numpy()
    pressure_predictions = standardized * preparation["target_scale"] + preparation["target_mean"]
    errors = regression_metrics(pressure_predictions, validation_answers)
    print(f"Practice epoch {epoch}: train MSE {total_loss / len(train):.4f}; "
          f"SBP MAE {errors['sbp_mae']:.2f}, DBP MAE {errors['dbp_mae']:.2f} mmHg")

## 7. Start a fresh, saved experiment

The runner below creates a **fresh** `FeatureMLP` and resets the seeds; it does not continue the practice model. It uses training-only preprocessing, early stopping, and restores the epoch with the best average SBP/DBP validation MAE. The defaults allow at most 100 epochs, with ten epochs of patience.

The training-mean predictor is a diagnostic comparison. Real model comparisons must use the same eligible windows and label version as the team's other models. No test scoring happens here.

In [ ]:
run_directory = ROOT / "data/derived/feature_mlp" / ("notebook_" + uuid4().hex[:10])
result, metrics = run_experiment(
    table, feature_names, label_version, dataset_version, run_directory,
    config=settings, demo=USE_DEMO,
    preparation_metadata=None if USE_DEMO else json.loads((DATA_PATH.parent / "preparation_summary.json").read_text()),
    data_sha256=file_sha256(DATA_PATH) if not USE_DEMO else None,
    feature_config_sha256=file_sha256(FEATURE_CONFIG_PATH) if not USE_DEMO else None,
)
print("Saved:", run_directory.relative_to(ROOT))
print("Best epoch:", result["best_epoch"])
print("ARTIFICIAL PROGRAMMING CHECK" if USE_DEMO else "DEVELOPMENT VALIDATION RESULTS")
display(metrics)
display(plot_history(result["history"], demo=USE_DEMO))

## 8. Reload and predict from features alone

The bundle saves the feature order, training statistics, settings, and weights. Future prediction needs those PPG features, not ABP or labels. Reloading should reproduce predictions; IDs remain available for matching results to source windows.

In [ ]:
restored, saved_preparation, metadata = load_bundle(run_directory / "model.pt")
feature_rows = validation[feature_names]
expected = predict(result["model"], feature_rows, result["preprocessing"])
reloaded = predict(restored, feature_rows, saved_preparation)
np.testing.assert_allclose(expected, reloaded, rtol=0, atol=1e-6)
print("Reloaded predictions match. Scope:", metadata["provenance"]["scope"])
display(pd.read_csv(run_directory / "validation_predictions.csv").head())

## Full comparison and interpretation

Use `experiments.runners.full_experiment` for the agreed six- versus fourteen-feature comparison, three training configurations, three selected-model seeds, mean/median baselines, Ridge diagnostic, and permutation importance. Each trial fits preprocessing using training rows only and restores its best validation checkpoint.

Feature importance measures reliance of the fitted model; it does not establish causation or clinical validity. Use the grouped importance alongside correlated individual measurements. Validation is a development set; reserve test evaluation for a later frozen experiment.